In [ ]:
#setup
import sys
import os
from math import log
import numpy as np
import pandas as pd
import scipy as sp
from PIL import Image
import matplotlib.pyplot as plt

import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.layers import Dense
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.callbacks import Callback,ReduceLROnPlateau,EarlyStopping


In [ ]:
# prepare the data

train_root_path = "../../Smartbin/data/trash_dataset"

from tensorflow.keras.preprocessing.image import ImageDataGenerator
batches = ImageDataGenerator().flow_from_directory(directory=train_root_path, target_size=(56,56) ,batch_size=20050)

In [ ]:
batches.class_indices

In [ ]:
imgs, labels = next(batches)
imgs.shape

In [ ]:
labels.shape

In [ ]:
classes = batches.class_indices.keys()
perc = (sum(labels)/labels.shape[0])*100

plt.xticks(rotation='vertical')
plt.bar(classes,perc)

In [ ]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(imgs/255.,labels, test_size=0.2)
x_train.shape
y_test.shape

In [ ]:
num_classes = len(classes)
# we do not need to apply one-hot encoding to the labels as in https://keras.io/examples/vision/swin_transformers/
# since the dataset data is already prepared for the multi-class classification
#y_train = keras.utils.to_categorical(y_train, num_classes)
#y_test = keras.utils.to_categorical(y_test, num_classes)
print(f"x_train shape: {x_train.shape} - y_train shape: {y_train.shape}")
print(f"x_test shape: {x_test.shape} - y_test shape: {y_test.shape}")

In [ ]:
!pip install keras_cv_attention_models

In [ ]:
from keras_cv_attention_models import coatnet
# requires 
input_shape=(56,56,3)  # (64,64,3) 
num_epochs = 200

batch_size = 32 # 128

learning_rate = 1e-4

num_epochs = 200
validation_split = 0.175
weight_decay = 0.0001
label_smoothing = 0.1

In [ ]:
base_model = coatnet.CoAtNet0(
    input_shape=input_shape, 
    num_classes=1000,  # Use 1000 to load the pre-trained weights
    drop_connect_rate=0.2,
    classifier_activation=None)  # No activation for the base model

# Add a new final layer for your specific number of classes
x = base_model.output
x = Dense(num_classes, activation='softmax', name='custom_predictions')(x)  # Ensure unique name
model = Model(inputs=base_model.input, outputs=x)

# Load pre-trained weights, ignoring the final layer
model.load_weights('/home/jupyter-iec_smartbin/.keras/models/coatnet0_224_imagenet.h5', by_name=True, skip_mismatch=True)


In [ ]:
model.compile(
    optimizer=Adam(learning_rate=learning_rate),
    loss=CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)

# Callbacks
early_stop = EarlyStopping(monitor='val_accuracy', patience=50, restore_best_weights=True)
lr_reduction = ReduceLROnPlateau(monitor='val_accuracy', patience=5, factor=0.5, min_lr=1e-6)

import time

class TimeHistory(Callback):
  def on_train_begin(self, logs=None):
    self.times = []
    self.epoch_start_time = time.time()  
  def on_epoch_end(self, epoch, logs=None):
    self.times.append(time.time() - self.epoch_start_time)
    self.epoch_start_time = time.time()
      
time_callback = TimeHistory()

model_fit = model.fit(
    x_train,
    y_train,
    epochs=num_epochs,
    validation_split=validation_split,
    verbose =1,
    callbacks=[early_stop,lr_reduction, time_callback]
)

In [ ]:
total_training_time = sum(time_callback.times)
print("Total training time:", total_training_time, "seconds")

In [ ]:
model.evaluate(x_test,  y_test)

In [ ]:
# plot the loss
plt.plot(model_fit.history['loss'], label='train loss')
plt.plot(model_fit.history['val_loss'], label='val loss')
plt.legend()
plt.show()
# plt.savefig('Swin_LossVal_loss.jpg',format='jpg')

plt.close()
# plot the accuracy
plt.plot(model_fit.history['accuracy'], label='train acc')
plt.plot(model_fit.history['val_accuracy'], label='val acc')
plt.legend()
plt.show()
# plt.savefig('Swin_AccVal_acc.jpg',format="jpg")

plt.close()

# Make predictions

In [ ]:
pred_x = model.predict(x_test, verbose=0) 
y_pred=np.argmax(pred_x,axis=1)
y_pred

In [ ]:
y_test2 = np.argmax(y_test, axis=1)
y_test2

In [ ]:
from sklearn import metrics
c_matrix = metrics.confusion_matrix(y_test2, y_pred)

In [ ]:
import seaborn as sns
def confusion_matrix(confusion_matrix, class_names, figsize = (10,7), fontsize=14):
    df_cm = pd.DataFrame(
        confusion_matrix, index=class_names, columns=class_names, 
    )
    fig = plt.figure(figsize=figsize)
    try:
        heatmap = sns.heatmap(df_cm, annot=True, fmt="d")
    except ValueError:
        raise ValueError("Confusion matrix values must be integers.")
    heatmap.yaxis.set_ticklabels(heatmap.yaxis.get_ticklabels(), rotation=0, ha='right', fontsize=fontsize)
    heatmap.xaxis.set_ticklabels(heatmap.xaxis.get_ticklabels(), rotation=45, ha='right', fontsize=fontsize)
    plt.ylabel('True label')
    plt.xlabel('Predicted label')

In [ ]:
class_names= batches.class_indices.keys()
confusion_matrix(c_matrix, class_names, figsize = (20,7), fontsize=14)

In [ ]:
from sklearn import metrics
# Print the precision and recall, among other metrics
report =  metrics.classification_report(y_test2, y_pred, digits=3, output_dict=True)

df = pd.DataFrame(report).transpose().reset_index()
df = df.rename(columns={"index": "class_label"})
df = pd.DataFrame(report).transpose()
df

In [ ]:
clf_rep = metrics.precision_recall_fscore_support(y_test2, y_pred)
out_dict = {
             "precision" : clf_rep[0].round(3)
            ,"recall" : clf_rep[1].round(3)
            ,"f1-score" : clf_rep[2].round(3)
            ,"support" : clf_rep[3]
            }
out_df = pd.DataFrame(out_dict).reset_index().rename(columns={"index": "class_label"})
class_label_values = dict(zip(range(0,len(batches.class_indices)), batches.class_indices))
out_df['class_label'] = out_df['class_label'].map(class_label_values)
out_df